# Stock Price Prediction with an LSTM

Companion code for the report.
Trains an LSTM model to predict the next day's close price for the AMC stock, using the last 12 days of OHLCV data as input. The model is trained on roughly 2 years of daily data fetched from Yahoo Finance via `yfinance`.


## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import yfinance as yf
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error


## Data collection

In [ ]:
ticker = 'AMC'
data = yf.download(ticker, period="2y", interval="1d")
data.head()


## Evolution of the stock price

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(data.index, data["Close"])
plt.title("Yearly evolution of stock price")
plt.xticks(rotation=90)
plt.xlabel("Date")
plt.ylabel("Price (USD)")
plt.grid(True)
plt.show()


## Preprocessing

Two `MinMaxScaler` instances are used so it's straightforward to inverse-transform the close price predictions back to dollars later.


In [ ]:
look_back = 12

close_scaler = MinMaxScaler(feature_range=(0, 1))
other_scaler = MinMaxScaler(feature_range=(0, 1))

scaled_close = close_scaler.fit_transform(data[['Close']])
scaled_others = other_scaler.fit_transform(data[['Open', 'Low', 'High', 'Volume']])
scaled_data = np.hstack([scaled_close, scaled_others])


## Build sliding-window samples and train/test split

In [ ]:
def create_data_and_labels(dataset, look_back=12):
    X, y = [], []
    for i in range(len(dataset) - look_back):
        X.append(dataset[i:i + look_back, :])
        y.append(dataset[i + look_back, 0])  # Only the close is predicted
    return np.array(X), np.array(y)

X, y = create_data_and_labels(scaled_data, look_back)

# 80/20 train/test split, preserving order (time series)
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# LSTM expects input in the form: (samples, time_steps, features)
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 5))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 5))


## Build and compile the LSTM

In [ ]:
tf.random.set_seed(42)
np.random.seed(42)

model = Sequential([
    LSTM(64, input_shape=(look_back, 5)),
    Dropout(0.2),
    Dense(1)
])

model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()


## Train the model

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=16,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)


## Loss over epochs

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Training History")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()


## Predictions and inverse scaling

In [ ]:
# Training and testing predictions
train_pred = model.predict(X_train, verbose=0)
test_pred = model.predict(X_test, verbose=0)

# Inverse transform predictions and actual values
train_pred_inv = close_scaler.inverse_transform(train_pred)
test_pred_inv = close_scaler.inverse_transform(test_pred)
y_train_inv = close_scaler.inverse_transform(y_train.reshape(-1, 1))
y_test_inv = close_scaler.inverse_transform(y_test.reshape(-1, 1))

# Sanity checks
print("train_pred range:", train_pred.min(), train_pred.max())
print("test_pred range:", test_pred.min(), test_pred.max())
print("y_train range:", y_train.min(), y_train.max())  # should be 0-1
print("model output shape:", train_pred.shape)         # should be (-1, 1)


## Predictions plot vs actual and 12-day moving average

In [ ]:
train_plot = np.full((len(scaled_data), 1), np.nan)
test_plot = np.full((len(scaled_data), 1), np.nan)

# Place predictions in the correct time positions
train_plot[look_back:look_back + len(train_pred_inv)] = train_pred_inv
test_plot[look_back + len(train_pred_inv):look_back + len(train_pred_inv) + len(test_pred_inv)] = test_pred_inv

actual_values = close_scaler.inverse_transform(scaled_data[:, 0].reshape(-1, 1))

plt.figure(figsize=(12, 5))
ma_12 = data['Close'].rolling(window=12).mean().values
plt.plot(data.index, ma_12, label="12-day MA", linestyle="--", color="orange")
plt.plot(data.index, actual_values, label="Actual")
plt.plot(data.index, train_plot, label="Train Prediction")
plt.plot(data.index, test_plot, label="Test Prediction")
plt.title("LSTM Forecasting: Actual vs Predicted")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.grid(True)
plt.show()


## Metrics

In [ ]:
train_rmse = np.sqrt(mean_squared_error(y_train_inv, train_pred_inv))
test_rmse = np.sqrt(mean_squared_error(y_test_inv, test_pred_inv))
train_mae = mean_absolute_error(y_train_inv, train_pred_inv)
test_mae = mean_absolute_error(y_test_inv, test_pred_inv)

print(f"Train RMSE: {train_rmse:.4f} | Train MAE: {train_mae:.4f}")
print(f"Test  RMSE: {test_rmse:.4f}  | Test  MAE: {test_mae:.4f}")


## Predict the next day's close price

In [ ]:
last_sequence = scaled_data[-look_back:]                  # shape (12, 5)
last_sequence = last_sequence.reshape(1, look_back, 5)    # shape (1, 12, 5)

next_day_scaled = model.predict(last_sequence, verbose=0)
next_day_price = close_scaler.inverse_transform(next_day_scaled)
print(f"Predicted next day close for {ticker}: ${next_day_price[0][0]:.2f}")
